In [1]:
import torch
import torch.nn as nn

In [2]:
class CNN1DAutoencoder(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(CNN1DAutoencoder, self).__init__()
        
        # Initialize dimensions
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # Encoder layers
        self.encoder = nn.Sequential(
            # First convolutional layer
            nn.Conv1d(in_channels=1, 
                     out_channels=16, 
                     kernel_size=3, 
                     stride=2, 
                     padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(16),
            
            # Second convolutional layer
            nn.Conv1d(in_channels=16, 
                     out_channels=32, 
                     kernel_size=3, 
                     stride=2, 
                     padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            
            # Third convolutional layer
            nn.Conv1d(in_channels=32, 
                     out_channels=hidden_size, 
                     kernel_size=3, 
                     stride=2, 
                     padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_size)
        )
        
        # Decoder layers
        self.decoder = nn.Sequential(
            # First transposed convolution
            nn.ConvTranspose1d(in_channels=hidden_size,
                              out_channels=32,
                              kernel_size=3,
                              stride=2,
                              padding=1,
                              output_padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            
            # Second transposed convolution
            nn.ConvTranspose1d(in_channels=32,
                              out_channels=16,
                              kernel_size=3,
                              stride=2,
                              padding=1,
                              output_padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(16),
            
            # Third transposed convolution
            nn.ConvTranspose1d(in_channels=16,
                              out_channels=1,
                              kernel_size=3,
                              stride=2,
                              padding=1,
                              output_padding=1),
            nn.Sigmoid()  # For normalized input data
        )
    
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        z = self.encode(x)
        return self.decode(z)

In [3]:
# Create sample data
input_size = 128
hidden_size = 64
batch_size = 32

In [4]:
# Initialize model
model = CNN1DAutoencoder(input_size=input_size, hidden_size=hidden_size)

In [5]:
model

CNN1DAutoencoder(
  (encoder): Sequential(
    (0): Conv1d(1, 16, kernel_size=(3,), stride=(2,), padding=(1,))
    (1): ReLU()
    (2): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Conv1d(16, 32, kernel_size=(3,), stride=(2,), padding=(1,))
    (4): ReLU()
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): Conv1d(32, 64, kernel_size=(3,), stride=(2,), padding=(1,))
    (7): ReLU()
    (8): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (decoder): Sequential(
    (0): ConvTranspose1d(64, 32, kernel_size=(3,), stride=(2,), padding=(1,), output_padding=(1,))
    (1): ReLU()
    (2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ConvTranspose1d(32, 16, kernel_size=(3,), stride=(2,), padding=(1,), output_padding=(1,))
    (4): ReLU()
    (5): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)

In [6]:
# Create random input data
x = torch.randn(batch_size, 1, input_size)
print(x)
print(x.shape)

tensor([[[ 0.1692, -0.2945,  0.0202,  ...,  0.0996, -1.1277,  1.8036]],

        [[ 0.1995, -0.0805,  1.1258,  ...,  0.4641,  0.2915, -0.1788]],

        [[-0.3546,  1.1542,  0.0326,  ..., -0.3709, -1.9479, -1.0947]],

        ...,

        [[ 0.8963, -0.9454,  0.4444,  ..., -1.1645, -0.5143, -0.6201]],

        [[-0.5330,  1.0663, -0.9595,  ...,  0.5369, -1.4926,  0.6091]],

        [[-1.1741, -0.2080,  0.2966,  ...,  0.0635,  1.4885, -0.5945]]])
torch.Size([32, 1, 128])


In [7]:
# Get encoded representation
encoded = model.encode(x)
print(f"Encoded shape: {encoded.shape}")

Encoded shape: torch.Size([32, 64, 16])


In [8]:
encoded

tensor([[[-6.6863e-01, -6.6863e-01, -3.6256e-01,  ..., -6.6863e-01,
           1.2492e+00, -6.6863e-01],
         [-1.2383e-01, -5.9135e-01, -1.3842e-01,  ...,  1.2319e+00,
          -5.9135e-01,  1.7082e+00],
         [-7.2039e-01,  5.7315e-01,  6.1453e-01,  ..., -7.2039e-01,
           6.5853e-02, -7.2039e-01],
         ...,
         [-1.1123e-01,  2.9291e-01, -5.6010e-01,  ...,  1.7991e-01,
           1.5665e+00, -5.6010e-01],
         [ 2.4330e+00, -5.8435e-01, -5.8435e-01,  ..., -5.8435e-01,
           3.3238e+00, -5.8435e-01],
         [ 3.1637e-01, -5.8068e-01, -5.8068e-01,  ..., -5.8068e-01,
          -5.8068e-01, -5.8068e-01]],

        [[-6.6863e-01, -6.6863e-01,  8.2696e-01,  ...,  2.4400e+00,
          -5.8559e-01,  2.1064e-01],
         [ 1.2475e-01, -5.9135e-01, -3.0318e-01,  ..., -5.9135e-01,
          -5.9135e-01,  3.5847e-01],
         [ 1.1072e-01, -3.6322e-01, -1.8644e-01,  ..., -7.2039e-01,
          -1.2691e-01, -7.2039e-01],
         ...,
         [-2.4624e-01,  4

In [9]:
# Get decoded (reconstructed) data
decoded = model.decode(encoded)
print(f"Decoded shape: {decoded.shape}")

Decoded shape: torch.Size([32, 1, 128])


In [10]:
decoded

tensor([[[0.2763, 0.7086, 0.5357,  ..., 0.3026, 0.3029, 0.8224]],

        [[0.3471, 0.1092, 0.5375,  ..., 0.2701, 0.5244, 0.2467]],

        [[0.4574, 0.4463, 0.2252,  ..., 0.4294, 0.5699, 0.6187]],

        ...,

        [[0.4432, 0.0627, 0.7629,  ..., 0.7169, 0.6696, 0.7938]],

        [[0.3202, 0.0988, 0.4754,  ..., 0.4658, 0.5079, 0.7335]],

        [[0.5638, 0.9530, 0.9580,  ..., 0.0567, 0.6070, 0.6692]]],
       grad_fn=<SigmoidBackward0>)

In [11]:
# Full forward pass
reconstructed = model(x)
print(f"Reconstructed shape: {reconstructed.shape}")

Reconstructed shape: torch.Size([32, 1, 128])
